# Module 5 · Vector Databases & Embedding Systems

**Track:** LLMOps  
**Toolchain:** ChromaDB · pgvector · FAISS · Sentence-Transformers  
**Objective:** Understand how vector databases work, how to choose embedding models, and how to build scalable semantic search — the foundation of RAG systems.

---

## 🧠 Why Vector Databases Before RAG?

You can't build a RAG pipeline without understanding WHERE and HOW your embeddings are stored and searched. This notebook teaches the foundation; the next notebook builds the full RAG pipeline on top.

### 🤔 What Are Embeddings?

An **embedding** is a list of numbers (a vector) that represents the MEANING of text. Similar texts have similar vectors.

```
"The cat sat on the mat"   → [0.21, -0.45, 0.87, ...]  ← 384 numbers
"A kitten was on the rug"  → [0.19, -0.43, 0.89, ...]  ← Very similar!
"Stock prices fell today"  → [-0.72, 0.33, -0.11, ...] ← Very different!
```

### 🤔 What Is a Vector Database?

A regular database stores rows and columns. A vector database stores **vectors** and finds the CLOSEST ones to a query vector.

| Regular Database | Vector Database |
|-----------------|----------------|
| `SELECT * WHERE name = 'John'` | `Find 5 vectors closest to this query vector` |
| Exact match | Approximate nearest neighbor (ANN) |
| Milliseconds | Milliseconds (with good indexing) |
| Scales to billions of rows | Scales to billions of vectors |

---

## ✅ Knowledge Check
### Q1: What makes vector embeddings better than keyword search for RAG?
<details><summary>Answer</summary>
Embeddings capture semantic meaning and relationships between concepts, allowing matches even without exact keyword overlap.
</details>
### Q2: Why use Approximate Nearest Neighbor (ANN) search instead of Exact KNN?
<details><summary>Answer</summary>
Exact KNN computes distances to every single vector, which scales linearly and is too slow for millions of vectors. ANN trades a tiny bit of absolute accuracy for massive performance gains.
</details>


## 🔨 Practical Practice
### Exercise 1: Embedding Comparison
Use a local embedding model to encode 5 different sentences and compute their cosine similarities to see semantic clustering.
### Exercise 2: Minimal FAISS
Build a minimal FAISS index and retrieve the top-2 most relevant text chunks for a sample user query.


## 📑 Table of Contents

* [🧠 Why Vector Databases Before RAG?](#why-vector-databases-before-rag)
  * [🤔 What Are Embeddings?](#what-are-embeddings)
  * [🤔 What Is a Vector Database?](#what-is-a-vector-database)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Embedding Models: Choosing the Right One](#1-embedding-models-choosing-the-right-one)
  * [🤔 Not All Embeddings Are Equal](#not-all-embeddings-are-equal)
  * [⚖️ Embedding Size Trade-off](#embedding-size-trade-off)
* [2 · Vector Database Comparison](#2-vector-database-comparison)
  * [The Landscape (2026)](#the-landscape-2026)
  * [🤔 How to Choose?](#how-to-choose)
* [3 · Indexing Algorithms: HNSW, IVF, and Beyond](#3-indexing-algorithms-hnsw-ivf-and-beyond)
  * [🤔 How Does Vector Search Find Similar Items So Fast?](#how-does-vector-search-find-similar-items-so-fast)
  * [HNSW Explained Simply](#hnsw-explained-simply)
* [4 · Fine-Tuning & Evaluating Embeddings](#4-fine-tuning-evaluating-embeddings)
  * [Evaluation Metrics: NDCG & MRR](#evaluation-metrics-ndcg-mrr)
  * [Hard Negative Mining](#hard-negative-mining)
  * [Matryoshka Representation Learning (MRL)](#matryoshka-representation-learning)
* [5 · Hybrid Search: Vector + Keyword](#5-hybrid-search-vector--keyword)
  * [🤔 Why Vector Search Alone Isn't Enough](#why-vector-search-alone-isnt-enough)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Decision Guide](#decision-guide)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Scanning 10 million documents linearly takes hours. Seniors use Vector Databases (Chroma, pgvector) with specialized Nearest-Neighbor indexes (HNSW) to find semantically related text in 5 milliseconds.

**Mental Model:** Embeddings convert philosophical concepts into GPS coordinates. A Vector Database is Google Maps calculating the shortest distance between your search query's coordinate and all document coordinates.

**Common Junior Pitfall:** Ignoring hybrid search. Vector search is great for 'concept' matching, but terrible at exact keyword matching (like finding a specific SKU number). Production systems combine Vector Search + BM25 Keyword Search.

---


In [ ]:
# Cell 1 — Install
!pip install -q chromadb sentence-transformers numpy scikit-learn

## 1 · Embedding Models: Choosing the Right One

### 🤔 Not All Embeddings Are Equal

| Model | Dimensions | Speed | Quality | Best For |
|-------|-----------|-------|---------|----------|
| **all-MiniLM-L6-v2** | 384 | ⚡ Very fast | Good | Prototyping, small datasets |
| **bge-small-en-v1.5** | 384 | ⚡ Fast | Better | Production with speed focus |
| **bge-large-en-v1.5** | 1024 | 🔄 Medium | High | Production with quality focus |
| **e5-large-v2** | 1024 | 🔄 Medium | High | Multilingual, enterprise |
| **text-embedding-3-large** | 3072 | 🐢 Slow (API) | Highest | Maximum quality, OpenAI API |
| **Cohere embed-v3** | 1024 | 🔄 Medium (API) | Very High | Enterprise, multilingual |

### ⚖️ Embedding Size Trade-off

| Dimension | Storage per 1M docs | Search Speed | Quality |
|-----------|--------------------|--------------|---------|
| 384 | ~1.5 GB | Fastest | Good |
| 768 | ~3 GB | Fast | Better |
| 1024 | ~4 GB | Medium | High |
| 3072 | ~12 GB | Slower | Highest |

**Rule of thumb:** Start with `all-MiniLM-L6-v2` (fast + free). Upgrade to `bge-large` or OpenAI if quality matters more than cost.

In [ ]:
# Cell 2 — Embedding model comparison
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Simulate different embedding models (in production, use sentence-transformers)
class EmbeddingModel:
    def __init__(self, name, dim, quality_factor=1.0):
        self.name = name
        self.dim = dim
        self.quality = quality_factor
    
    def embed(self, text):
        """Generate a deterministic embedding for demonstration."""
        np.random.seed(hash(text) % 2**32)
        vec = np.random.randn(self.dim) * self.quality
        return vec / np.linalg.norm(vec)
    
    def storage_per_million(self):
        """GB needed to store 1M embeddings."""
        return self.dim * 4 * 1_000_000 / (1024**3)  # float32

models = [
    EmbeddingModel('all-MiniLM-L6-v2', 384, 0.9),
    EmbeddingModel('bge-large-en-v1.5', 1024, 0.95),
    EmbeddingModel('text-embedding-3-large', 3072, 1.0),
]

test_pairs = [
    ('machine learning optimization', 'ML model training techniques'),          # Similar
    ('machine learning optimization', 'best restaurants in Paris'),             # Different
    ('how to deploy a model', 'model deployment strategies'),                    # Similar
]

print('📊 Embedding Model Comparison')
print('=' * 70)
for model in models:
    print(f'\n  {model.name} (dim={model.dim}, storage={model.storage_per_million():.1f} GB/1M docs):')
    for text_a, text_b in test_pairs:
        vec_a = model.embed(text_a).reshape(1, -1)
        vec_b = model.embed(text_b).reshape(1, -1)
        sim = cosine_similarity(vec_a, vec_b)[0][0]
        bar = '█' * int(abs(sim) * 20)
        print(f'    sim={sim:+.3f} {bar} | "{text_a[:30]}" vs "{text_b[:30]}"')

print(f'\n💡 Higher dimensions = better quality but more storage and slower search.')

---

## 2 · Vector Database Comparison

### The Landscape (2026)

| Database | Type | Max Scale | Unique Strength | Best For |
|----------|------|----------|-----------------|----------|
| **ChromaDB** | Embedded | ~10M vectors | Simplest API, runs in-process | Prototyping, small projects |
| **pgvector** | Extension | ~50M vectors | Uses existing PostgreSQL | Teams already using Postgres |
| **Weaviate** | Standalone | ~1B vectors | Built-in ML models, hybrid search | Production semantic search |
| **Pinecone** | Managed SaaS | ~1B vectors | Zero ops, auto-scaling | Teams that want managed infra |
| **Qdrant** | Standalone | ~1B vectors | Advanced filtering, Rust performance | High-performance filtering |
| **Milvus** | Standalone | ~10B vectors | Largest scale, GPU acceleration | Massive datasets |
| **FAISS** | Library | ~1B vectors | Facebook's library, fastest search | When you need raw speed |

### 🤔 How to Choose?

```
Are you prototyping?
  → Yes: ChromaDB (5 lines of code to start)
  → No: Continue...

Do you already use PostgreSQL?
  → Yes: pgvector (add vector search to existing DB)
  → No: Continue...

Do you want to manage infrastructure?
  → No: Pinecone (fully managed)
  → Yes: Continue...

Do you need advanced filtering?
  → Yes: Qdrant or Weaviate
  → No: Milvus (if > 100M vectors) or Weaviate (general)
```

In [ ]:
# Cell 3 — ChromaDB hands-on: building a vector store
import chromadb
import numpy as np

# Create an in-memory vector store
client = chromadb.Client()
collection = client.create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

# Add documents with metadata
documents = [
    {"text": "Q4 2025 revenue was $58.1M, up 15% year over year.", "source": "finance", "quarter": "Q4"},
    {"text": "The engineering team grew from 45 to 67 engineers in 2025.", "source": "hr", "quarter": "yearly"},
    {"text": "Customer churn rate decreased from 5.2% to 3.8%.", "source": "product", "quarter": "Q4"},
    {"text": "AWS infrastructure costs increased 22% due to GPU expansion.", "source": "finance", "quarter": "Q4"},
    {"text": "New product launch planned for Q1 2026 targeting enterprise.", "source": "product", "quarter": "Q1"},
    {"text": "Employee satisfaction score improved from 7.2 to 8.1 out of 10.", "source": "hr", "quarter": "yearly"},
    {"text": "Model latency reduced from 450ms to 120ms after vLLM migration.", "source": "engineering", "quarter": "Q3"},
    {"text": "Total funding raised: $25M Series B led by Sequoia Capital.", "source": "finance", "quarter": "Q2"},
]

collection.add(
    documents=[d["text"] for d in documents],
    metadatas=[{"source": d["source"], "quarter": d["quarter"]} for d in documents],
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print(f"✅ Loaded {collection.count()} documents into ChromaDB")

# Query 1: Simple semantic search
print("\n🔍 Query 1: 'What are the financial results?'")
results = collection.query(query_texts=["financial results and revenue"], n_results=3)
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    sim = 1 - dist
    print(f"  [{sim:.3f}] [{meta['source']}] {doc}")

# Query 2: Filtered search (only finance docs)
print("\n🔍 Query 2: 'costs' — filtered to source=finance only")
results = collection.query(
    query_texts=["costs and expenses"], 
    n_results=3,
    where={"source": "finance"}
)
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    sim = 1 - dist
    print(f"  [{sim:.3f}] [{meta['source']}] {doc}")

print(f"\n💡 Metadata filtering narrows results BEFORE vector search.")
print(f"   This is called 'hybrid search' — combining vector + filter.")

---

## 3 · Indexing Algorithms: HNSW, IVF, and Beyond

### 🤔 How Does Vector Search Find Similar Items So Fast?

**Brute force:** Compare the query vector to every stored vector. Works, but O(n) — too slow for millions of vectors.

**Indexing algorithms** create shortcuts so you only compare against a small subset.

| Algorithm | How It Works | Speed | Accuracy | Memory |
|-----------|-------------|-------|----------|--------|
| **Flat (brute force)** | Compare to every vector | 🐢 O(n) | 100% | Low |
| **IVF** | Cluster vectors, search nearest clusters | 🚗 Fast | 95-99% | Low |
| **HNSW** | Graph-based, navigable layers | 🚀 Very fast | 97-99% | High (2-3x) |
| **SCANN** | Quantize + asymmetric search | 🚀 Very fast | 95-99% | Medium |

### HNSW Explained Simply

Think of finding a friend in a city:
- **Brute force:** Visit every house and check if your friend is there
- **HNSW:** Start at a high-level map (neighborhoods), zoom into the right neighborhood, then the right street, then the right house

```
Layer 3 (few nodes):    A ──── B ──── C           ← Start here (coarse)
Layer 2 (more nodes):   A─D─B──E──F─C─G           ← Navigate down
Layer 1 (most nodes):   A D B H E I F C G J K     ← Find nearest neighbors
```

In [ ]:
# Cell 4 — Compare search strategies
import numpy as np
import time

def brute_force_search(query, vectors, top_k=5):
    """O(n) search — compare to every vector."""
    sims = vectors @ query
    top_indices = np.argsort(sims)[-top_k:][::-1]
    return [(i, sims[i]) for i in top_indices]

def ivf_search(query, vectors, cluster_centers, top_k=5, nprobe=3):
    """IVF: search only the nearest clusters."""
    # Find nearest cluster centers
    center_sims = cluster_centers @ query
    probe_clusters = np.argsort(center_sims)[-nprobe:][::-1]
    
    # Search only within those clusters
    cluster_size = len(vectors) // len(cluster_centers)
    candidates = []
    for c in probe_clusters:
        start = c * cluster_size
        end = min(start + cluster_size, len(vectors))
        for i in range(start, end):
            candidates.append((i, vectors[i] @ query))
    
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

# Generate test data
np.random.seed(42)
n_vectors = 100_000
dim = 384
vectors = np.random.randn(n_vectors, dim).astype(np.float32)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
query = np.random.randn(dim).astype(np.float32)
query = query / np.linalg.norm(query)

# Cluster centers for IVF
n_clusters = 100
cluster_centers = vectors[::n_vectors // n_clusters][:n_clusters]

# Benchmark
print(f'📊 Search Strategy Benchmark ({n_vectors:,} vectors, {dim} dimensions)')
print('=' * 60)

# Brute force
start = time.time()
bf_results = brute_force_search(query, vectors)
bf_time = time.time() - start
print(f'  Brute Force:  {bf_time*1000:>8.2f} ms | Top result: idx={bf_results[0][0]}, sim={bf_results[0][1]:.4f}')

# IVF
start = time.time()
ivf_results = ivf_search(query, vectors, cluster_centers, nprobe=3)
ivf_time = time.time() - start
print(f'  IVF (nprobe=3): {ivf_time*1000:>6.2f} ms | Top result: idx={ivf_results[0][0]}, sim={ivf_results[0][1]:.4f}')

# Recall: how many of IVF's results match brute force?
bf_ids = set(r[0] for r in bf_results)
ivf_ids = set(r[0] for r in ivf_results)
recall = len(bf_ids & ivf_ids) / len(bf_ids)

print(f'\n  Speedup: {bf_time/ivf_time:.1f}x faster')
print(f'  Recall:  {recall:.0%} (how many true top-5 were found)')
print(f'\n💡 IVF trades a small amount of accuracy for much faster search.')
print(f'   HNSW (used by ChromaDB, Weaviate, Qdrant) is even faster with higher recall.')

---

## 4 · Fine-Tuning & Evaluating Embeddings

Using off-the-shelf models like `all-MiniLM-L6-v2` works for general English. But what if you are building search for highly specialized medical records or proprietary legal contracts? You need to *fine-tune* the embedding space so domain-specific concepts map correctly.

### Evaluation Metrics: NDCG & MRR

Before fine-tuning, you must evaluate current performance.
- **MRR (Mean Reciprocal Rank):** Evaluates if the *first relevant document* appeared near the top. Score is $1/\text{rank}$. If it's at position 3, MRR = 0.33.
- **NDCG (Normalized Discounted Cumulative Gain):** Evaluates the entire ranking order. It rewards putting highly relevant docs at rank 1, moderately relevant at rank 2, etc., penalizing late appearances logarithmically.

### Hard Negative Mining

To teach an embedding model the difference between highly nuanced documents, you contrastive-train it using **Triplets**: (Anchor, Positive, Negative).
- **Anchor:** "How to reset password"
- **Positive:** "To reset your password, visit the auth portal."
- **Random Negative:** "The cafeteria serves pasta on Tuesdays."
- **HARD Negative:** "To change your username, visit the auth portal."

The model easily pushes the Random Negative away. It needs **Hard Negatives** (similar wording, different semantic intent) to dramatically improve accuracy.

### Matryoshka Representation Learning (MRL)

A revolutionary embedding trick (used in OpenAI's `text-embedding-3`). Instead of treating the 1024-dimension vector equally, the model is trained so that the **first 256 dimensions contain 95% of the meaning**. 

You can natively truncate a 1024-d Matryoshka embedding down to 256-d to save 75% vector database storage cost, with almost zero loss in NDCG!


In [ ]:
# Cell 4.1 — Simulating Evaluating retrieval with MRR & NDCG
import numpy as np
from sklearn.metrics import ndcg_score

def calculate_mrr(relevance_array):
    """Returns 1/rank of the *first* relevant document."""
    for index, is_rel in enumerate(relevance_array):
        if is_rel > 0:
            return 1.0 / (index + 1)
    return 0.0

print("📊 Information Retrieval Metrics Evaluation")
print('=' * 60 + '\n')
# Array represents relevance of retrieved docs (1=relevant, 0=not)
# Perfect retrieval: relevant docs are at position 0
queries = [
    {"name": "Perfect Retrieval",          "rels": [1, 0, 0, 0, 0]},
    {"name": "Good (Found at rank 2)",   "rels": [0, 1, 0, 0, 0]},
    {"name": "Poor (Found at rank 5)",   "rels": [0, 0, 0, 0, 1]},
    {"name": "Failed (Not in top 5)",    "rels": [0, 0, 0, 0, 0]},
]

for q in queries:
    mrr = calculate_mrr(q['rels'])
    # For NDCG, we need true relevances and predicted scores.
    # We simulate predicted scores as strictly descending.
    true_relevance = np.asarray([q['rels']])
    pred_scores = np.asarray([[0.9, 0.7, 0.5, 0.3, 0.1]])
    
    if sum(q['rels']) == 0: ndcg = 0.0
    else: ndcg = ndcg_score(true_relevance, pred_scores)
    
    print(f"{q['name']:<25} | MRR: {mrr:.2f} | NDCG: {ndcg:.2f}")

print("\n💡 Key Insight: MRR cares only about the FIRST hit. NDCG cares about entire ranking quality.")


---

## 5 · Hybrid Search: Vector + Keyword

### 🤔 Why Vector Search Alone Isn't Enough

Vector search finds SEMANTICALLY similar results. But sometimes you need EXACT matches:

| Query | Vector Search Returns | What User Actually Wants |
|-------|---------------------|--------------------------|
| "error code E-4021" | "common error codes in the system" | The EXACT doc about E-4021 |
| "John Smith account" | "managing user accounts" | John Smith's SPECIFIC account |
| "CVE-2024-1234" | "security vulnerabilities overview" | The EXACT CVE entry |

**Hybrid search** = Vector search (semantic) + BM25/keyword search, with results fused.

| Strategy | Semantic Understanding | Exact Match | Best For |
|----------|----------------------|-------------|----------|
| Vector only | ✅ Excellent | ❌ Poor | Natural language questions |
| Keyword only (BM25) | ❌ Poor | ✅ Excellent | Known entity lookup |
| **Hybrid** | ✅ Excellent | ✅ Good | Production RAG systems |

In [ ]:
# Cell 5 — Hybrid search implementation
import re
from collections import Counter

class HybridSearch:
    """Combine vector similarity (semantic) with BM25 (keyword) search."""
    
    def __init__(self, alpha=0.7):
        self.alpha = alpha  # Weight: alpha=1 → pure vector, alpha=0 → pure keyword
        self.documents = []
        self.vectors = []
    
    def add(self, text, vector):
        self.documents.append(text)
        self.vectors.append(vector)
    
    def bm25_score(self, query, document, k1=1.5, b=0.75):
        """Simple BM25 keyword scoring."""
        query_terms = query.lower().split()
        doc_terms = document.lower().split()
        doc_len = len(doc_terms)
        avg_len = sum(len(d.split()) for d in self.documents) / len(self.documents)
        term_freq = Counter(doc_terms)
        
        score = 0
        for term in query_terms:
            tf = term_freq.get(term, 0)
            idf = 1.0  # Simplified
            score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_len))
        return score
    
    def search(self, query, query_vector, top_k=5):
        """Hybrid search: combine vector + keyword scores."""
        results = []
        for i, (doc, vec) in enumerate(zip(self.documents, self.vectors)):
            vec_score = np.dot(query_vector, vec)  # Cosine similarity
            kw_score = self.bm25_score(query, doc)
            
            # Normalize scores to [0, 1]
            vec_norm = (vec_score + 1) / 2
            kw_norm = min(kw_score / 5, 1.0)
            
            hybrid_score = self.alpha * vec_norm + (1 - self.alpha) * kw_norm
            results.append({'doc': doc, 'score': hybrid_score, 'vec': vec_norm, 'kw': kw_norm})
        
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]

# Set up hybrid search
hybrid = HybridSearch(alpha=0.6)  # 60% vector, 40% keyword

np.random.seed(42)
docs = [
    "Error code E-4021: Database connection timeout after 30 seconds.",
    "Common database errors include timeout, connection refused, and deadlock.",
    "CVE-2024-1234: Critical SQL injection vulnerability in PostgreSQL.",
    "Security best practices for database management.",
    "Troubleshooting guide for database performance issues.",
]
for doc in docs:
    vec = np.random.randn(384).astype(np.float32)
    vec = vec / np.linalg.norm(vec)
    hybrid.add(doc, vec)

query = "error E-4021"
query_vec = np.random.randn(384).astype(np.float32)
query_vec = query_vec / np.linalg.norm(query_vec)

print(f'🔍 Hybrid Search: "{query}"')
print(f'   Weights: {hybrid.alpha:.0%} vector + {1-hybrid.alpha:.0%} keyword')
print('=' * 70)
for r in hybrid.search(query, query_vec, top_k=3):
    print(f'  Score: {r["score"]:.3f} (vec={r["vec"]:.3f}, kw={r["kw"]:.3f})')
    print(f'    → {r["doc"]}')

print(f'\n💡 The keyword component boosts the exact E-4021 match that')
print(f'   vector search alone might miss.')

---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Why It Matters |
|---------|-----------------|----------------|
| Embeddings | Text → numbers that capture meaning | Foundation of semantic search |
| Embedding models | Size/quality trade-offs | Choose the right model for your use case |
| Vector DBs | ChromaDB, pgvector, Weaviate, etc. | Store and search embeddings at scale |
| Indexing (HNSW, IVF) | How fast search works | Understand the speed-accuracy trade-off |
| Hybrid search | Vector + keyword combined | Production RAG needs both |

### 🧠 Decision Guide

1. **Prototyping?** → ChromaDB (simplest, in-process)
2. **Already on Postgres?** → pgvector (add vectors to existing DB)
3. **Production, < 10M vectors?** → Weaviate or Qdrant
4. **Production, > 100M vectors?** → Milvus
5. **Don't want to manage infra?** → Pinecone (managed)

**Next →** `29_rag_pipeline.ipynb` — End-to-End RAG Pipeline: From Documents to Answers